# 06 — Reproducibility Check

Verifies all required artefacts from every phase are present and confirms the full pipeline produces consistent results.

**To run the full pipeline from scratch:**
```bash
python run_all.py
```
This executes notebooks 01–05 in order using nbconvert, then runs the artefact checks below.

**This notebook** performs the artefact checks interactively.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT    = Path('..').resolve()
PASS    = '  [OK]     '
FAIL    = '  [MISSING]'

---
## Check 1 — Source files

In [ ]:
src_files = ['src/api.py', 'src/features.py', 'src/model.py', 'requirements.txt']
failures = []

for f in src_files:
    exists = (ROOT / f).exists()
    print(f"{PASS if exists else FAIL} {f}")
    if not exists:
        failures.append(f)

assert not failures, f"Missing source files: {failures}"

---
## Check 2 — Cleaned data

In [ ]:
cleaned_files = [
    'data/cleaned/pit_stops_clean.csv',
    'data/cleaned/lap_times_clean.csv',
    'data/cleaned/results_clean.csv',
    'data/features.parquet',
]
failures = []

for f in cleaned_files:
    exists = (ROOT / f).exists()
    print(f"{PASS if exists else FAIL} {f}")
    if not exists:
        failures.append(f)

assert not failures, f"Missing data files: {failures}"

---
## Check 3 — EDA reports

In [ ]:
eda_reports = [
    'data/eda_reports/pit_stops_eda_report.md',
    'data/eda_reports/lap_times_eda_report.md',
    'data/eda_reports/results_eda_report.md',
]
failures = []

for f in eda_reports:
    exists = (ROOT / f).exists()
    print(f"{PASS if exists else FAIL} {f}")
    if not exists:
        failures.append(f)

assert not failures, f"Missing EDA reports: {failures}"

---
## Check 4 — Visuals (all 8 charts)

In [ ]:
visuals = [
    'visuals/pitstop_lap_distribution.png',
    'visuals/team_stop_duration.png',
    'visuals/strategy_frequency_by_season.png',
    'visuals/undercut_success_by_circuit.png',
    'visuals/position_change_distribution.png',
    'visuals/position_change_heatmap.png',
    'visuals/roc_curves.png',
    'visuals/shap_summary_plot.png',
]
failures = []

for f in visuals:
    exists = (ROOT / f).exists()
    print(f"{PASS if exists else FAIL} {f}")
    if not exists:
        failures.append(f)

assert not failures, f"Missing visuals: {failures}"

---
## Check 5 — Feature matrix integrity

In [ ]:
features = pd.read_parquet(ROOT / 'data' / 'features.parquet')

REQUIRED_COLS = [
    'stop_lap_pct', 'gap_to_car_ahead', 'is_undercut_attempt',
    'compound_hardness', 'team_avg_stop_time', 'prior_stops',
    'circuit_type', 'position_gained',
]

for col in REQUIRED_COLS:
    assert col in features.columns, f"Missing column: {col}"
    nulls = features[col].isnull().sum()
    status = 'OK' if nulls == 0 else f'NULLS: {nulls}'
    print(f"  [{status:^8}] {col}")

assert features['position_gained'].isin([0, 1]).all(), "position_gained contains non-binary values"
print(f"\nFeature matrix: {features.shape[0]:,} rows × {features.shape[1]} columns")
print(f"Target balance: {features['position_gained'].mean():.1%} positive class")

---
## Check 6 — Seeds are fixed (determinism)

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from src.model import train_evaluate

RANDOM_STATE = 42
FEATURE_COLS = ['stop_lap_pct', 'gap_to_car_ahead', 'is_undercut_attempt',
                'compound_hardness', 'team_avg_stop_time', 'prior_stops']

df = features.dropna(subset=['position_gained'])
X = df[FEATURE_COLS]
y = df['position_gained'].astype(int)

# Run the split twice — must produce identical results
splits = []
for _ in range(2):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    m = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
    splits.append(train_evaluate(m, X_tr, X_te, y_tr, y_te)['accuracy'])

assert splits[0] == splits[1], "Non-deterministic results detected — check random seeds"
print(f"Determinism check passed: both runs produced accuracy = {splits[0]}")

---
## Phase 6 complete

In [ ]:
print("=" * 50)
print("ALL CHECKS PASSED")
print("Phase 6 complete — project is fully reproducible.")
print("=" * 50)